In [0]:
from pyspark.sql.functions import col

# 1. Inicjalizacja tabeli Silver (jeśli nie istnieje)
spark.sql("""
CREATE TABLE IF NOT EXISTS dbw_showcase.default.payroll_silver (
    transaction_id STRING,
    emp_id INT,
    office_location STRING,
    hours_logged DOUBLE,
    timestamp TIMESTAMP,
    ingested_at TIMESTAMP
)
USING DELTA
""")

# Ścieżka na checkpoint dla strumienia Silver (śledzi, co już przetworzyliśmy)
CHECKPOINT_SILVER = "/Volumes/dbw_showcase/default/showcase_raw/checkpoints_payroll_silver"

# 2. Definiujemy funkcję dla foreachBatch (Standard korporacyjny dla upsertów)
def upsert_to_silver(microBatchDF, batchId):
    # Krok A: Deduplikacja w locie za pomocą PySparka (usuwamy duplikaty po transaction_id)
    deduplicated_df = microBatchDF.dropDuplicates(["transaction_id"])
    
    # Tworzymy widok tymczasowy dla SQL
    deduplicated_df.createOrReplaceTempView("payroll_updates")
    
    # Krok B: Idempotentny MERGE INTO (Zapis / Aktualizacja)
    microBatchDF.sparkSession.sql("""
        MERGE INTO dbw_showcase.default.payroll_silver target
        USING payroll_updates source
        ON target.transaction_id = source.transaction_id
        WHEN MATCHED THEN
            UPDATE SET 
                target.hours_logged = source.hours_logged,
                target.timestamp = source.timestamp
        WHEN NOT MATCHED THEN
            INSERT (transaction_id, emp_id, office_location, hours_logged, timestamp, ingested_at)
            VALUES (source.transaction_id, source.emp_id, source.office_location, source.hours_logged, source.timestamp, source.ingested_at)
    """)

# 3. Odczyt strumieniowy z Bronze i zapis do Silver
print("⏳ Uruchamianie strumieniowego UPSERTu do warstwy Silver...")

payroll_bronze_stream = spark.readStream.table("dbw_showcase.default.payroll_bronze")

query = (payroll_bronze_stream.writeStream
    .foreachBatch(upsert_to_silver)
    .option("checkpointLocation", CHECKPOINT_SILVER)
    .trigger(availableNow=True) # Optymalizacja kosztów: zjedz zaległości z Bronze i wyłącz
    .start()
)

query.awaitTermination()
print("✅ Strumień Payroll został zdeduplikowany i zapisany w warstwie Silver!")